# OPRD-100 scoring examples

OPRD-100 has two scoring workflows with different purposes. They should not be used interchangeably.

| Workflow | Input | Reaction pairing | Reported result |
|---|---|---|---|
| **Strict validation** | Human re-extraction used in the paper | Exact `(Reference, Location.Type, Location.Num)` | Mean similarity over matched validation reactions |
| **Lenient automated-extraction scoring** | LLM, OCR, or rule-based extraction | Content-based Hungarian matching within each paper and location type | Quality, coverage, and `quality × coverage` |

The strict workflow checks whether a human can reproduce the curated data while revisiting the same source locations. Automated systems often extract the correct chemistry but assign different Scheme/Table/Experimental labels or numbers, so they must use the lenient workflow.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Support launching Jupyter from either the repository root or notebooks/.
notebook_dir = Path.cwd()
if (notebook_dir / "notebooks").is_dir():
    notebook_dir = notebook_dir / "notebooks"
os.chdir(notebook_dir)
sys.path.insert(0, str(notebook_dir.parent / "src"))

from lenient_scoring import score_lenient
from validation import DataComparer

human_validation_path = notebook_dir.parent / "data" / "validation_reactions.json"

# Set this to a plain JSON list of OPRD-schema reaction records produced by an
# automated extractor. Leave as None to run only the bundled strict example.
automated_extraction_path = None  # e.g. Path("/path/to/automated_reactions.json")

## 1. Strict scoring: paper validation set

This reproduces the validation methodology used for the accompanying paper. Reactions are eligible to match only when their reference, location type, and location number agree exactly with OPRD-100. The reaction-SMILES metric uses exact InChIKey-set overlap and keeps stereochemistry.

Use this workflow **only for the human re-extraction validation set**, whose annotator revisited the same source locations. It is not the automated-extraction leaderboard score.

In [ ]:
strict = DataComparer(str(human_validation_path), similarity_method="inchikey")

strict_by_type = {
    "Experimental": strict.compute_comparison_scores(
        strict.val_experimental_data, strict.oprd_experimental_data
    ),
    "Table": strict.compute_comparison_scores(strict.val_table_data, strict.oprd_table_data),
    "Scheme": strict.compute_comparison_scores(strict.val_scheme_data, strict.oprd_scheme_data),
}
strict_combined = pd.concat(strict_by_type.values(), ignore_index=True)

print(f"Strict validation reactions matched: {len(strict_combined)}")

### Strict validation results

The headline result is the mean of `total_similarity` across the matched human-validation reactions. The table also exposes every component score so the paper result is reproducible rather than reduced to one number.

In [ ]:
metric_columns = [
    "reaction_smiles_similarity",
    "reaction_steps_similarity",
    "yield_similarity",
    "reagent_name_similarity",
    "reagent_amount_similarity",
    "solvent_similarity",
    "time_similarity",
    "temperature_similarity",
    "total_similarity",
]

strict_summary = pd.DataFrame(
    {
        location_type: {
            **results[metric_columns].mean().to_dict(),
            "matched_reactions": len(results),
        }
        for location_type, results in {
            **strict_by_type,
            "Combined": strict_combined,
        }.items()
    }
).T
strict_summary.round(3)

## 2. Lenient scoring: automated extraction

Automated extractors are scored by content because their location labels frequently differ from the curated labels even when the extracted chemistry is correct. The scorer uses Hungarian assignment within each paper and primary location type, excludes Figure reactions by default, and reports:

- **Quality**: mean similarity over matched reaction pairs.
- **Coverage**: matched reactions divided by in-scope gold reactions in attempted papers.
- **Score**: quality multiplied by coverage.

The automated leaderboard currently uses exact InChIKey-set SMILES similarity so molecular identity remains directly comparable with strict validation. Set `similarity_method="tanimoto"` to investigate graded molecular similarity, but do not mix that result with the official InChIKey leaderboard.

In [ ]:
if automated_extraction_path is None:
    automated_score = None
    print("Set automated_extraction_path in Cell 2 to run lenient scoring.")
else:
    automated_score = score_lenient(
        automated_extraction_path,
        similarity_method="inchikey",
        strip_stereo=False,
        type_aware_matching=True,
        exclude_types=["Figure"],
    )
    print("Lenient automated-extraction scoring complete.")

In [ ]:
if automated_score is not None:
    automated_summary = pd.Series(
        {
            "score_quality_x_coverage": automated_score.combined_mean_covered,
            "quality": automated_score.combined_mean,
            "coverage": automated_score.coverage,
            "matched_reactions": automated_score.num_matched,
            "in_scope_gold_reactions": automated_score.num_gold,
            "extracted_reactions": automated_score.num_extracted,
            "reaction_smiles_similarity": automated_score.reaction_smiles_mean,
            "yield_similarity": automated_score.yield_mean,
            "reagent_name_similarity": automated_score.reagent_name_mean,
            "solvent_similarity": automated_score.solvent_mean,
            "time_similarity": automated_score.time_mean,
            "temperature_similarity": automated_score.temperature_mean,
        },
        name="Automated extraction",
    )
    display(automated_summary.to_frame().round(3))
    display(pd.DataFrame(automated_score.per_type_breakdown()).T.round(3))

## Command-line equivalents

Strict human validation:

```bash
python scripts/run_scoring.py \
  --submission-file data/submissions/<human-validation-submission>.json \
  --output-dir results/strict_validation
```

Lenient automated extraction:

```bash
python scripts/run_scoring.py --lenient \
  --submission-file data/submissions/<automated-submission>.json \
  --output-dir results/automated_extraction
```

The GitHub Actions submission workflow runs the second command. Strict scoring is retained only to reproduce the paper's human-validation analysis.